# PolEmo2 — Stałe reprezentacje i dostrajanie enkodera

W kolejnych komórkach znajdziesz objaśnienia, kod i wyniki doświadczeń.
Czytaj je po kolei i uruchamiaj kod przyciskiem ▶ albo **Shift+Enter**.
Definicje funkcji przygotowują doświadczenie; obliczenia rozpocznie
dopiero komórka **Uruchomienie porównania** pod koniec notatnika.

Uczymy model rozpoznawać wydźwięk polskich opinii. W jednym doświadczeniu
używamy całego zbioru. W drugim uczymy model na tekstach o hotelach
i testujemy go na tekstach medycznych. Każdy model dopasowujemy osobno.

Wybierz **Plik → Zapisz kopię na Dysku**.
Do dostrajania wybierz **Środowisko wykonawcze → Zmień typ środowiska wykonawczego → GPU T4**, jeśli jest dostępne. Bezpłatny Colab nie gwarantuje dostępu do GPU. Na CPU kod także działa, ale dostrajanie trwa dłużej.
Materiały, dane i obliczenia nie wymagają płatnych usług ani klucza API.

[Artykuł i materiały](https://neuronovo.si/projekty/polemo2/).


In [ ]:
import importlib.util
import subprocess
import sys

# Colab udostępnia większość bibliotek. Instalujemy tylko brakujące,
# z oficjalnego rejestru PyPI; nie zmieniamy całego środowiska.
pakiety = {'numpy': 'numpy', 'pandas': 'pandas', 'sklearn': 'scikit-learn', 'matplotlib': 'matplotlib', 'torch': 'torch', 'transformers': 'transformers', 'safetensors': 'safetensors'}
brakujace = [nazwa_pypi for modul, nazwa_pypi in pakiety.items()
             if importlib.util.find_spec(modul) is None]
if brakujace:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *brakujace])
print("Biblioteki są dostępne.")


## Biblioteki, źródła i nazwy metod

Poniżej widać importy i stałe: wersję danych, wersję modelu, źródło
polskich wektorów fastText oraz podziały obu doświadczeń. Nie pobieramy
najnowszej, zmieniającej się wersji modelu: wskazujemy konkretną rewizję.
`ORIGINAL_MODULE_SHA256` dokumentuje źródło tej implementacji.
Pole `module_sha256` w raporcie odnosi się do tego oryginalnego pliku,
z którego funkcje rozdzielono na komórki; nie jest sumą całego notatnika.


In [ ]:
from __future__ import annotations
import copy
import gc
import hashlib
import importlib.metadata
import json
import os
import platform
import random
import re
import shutil
import sys
import time
import urllib.request
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import FeatureUnion, make_pipeline
from sklearn.preprocessing import StandardScaler
from types import SimpleNamespace

ORIGINAL_MODULE_SHA256 = "44b4be5cbf87b2d85696d5e6c7bbb6109ced6816a687695c85296be7376ef015"

SEED = 42

DATASET_ID = "clarin-pl/polemo2-official"

DATASET_REVISION = "802e35d2b12bae84bb07911d841e8f046dc2fcef"

ENCODER_ID = "sdadas/polish-distilroberta"

ENCODER_REVISION = "849b664fa3134beae84095d28a184c145c6a3aa5"

FASTTEXT_URL = "https://dl.fbaipublicfiles.com/fasttext/vectors-wiki/wiki.pl.vec"

LABELS = ["zero", "minus", "plus", "amb"]

METHOD_NAMES = {
    "dummy": "Najczęstsza klasa",
    "tfidf_nb": "TF-IDF słów + naiwny klasyfikator Bayesa",
    "tfidf_lr": "TF-IDF słów + regresja logistyczna",
    "tfidf_word_char_lr": "TF-IDF słów i fragmentów słów + regresja logistyczna",
    "fasttext_lr": "Średnia wektorów fastText PL + regresja logistyczna",
    "encoder_frozen_lr": "Zamrożony polski encoder + regresja logistyczna",
    "encoder_finetuned": "Dostrajanie polskiego encodera",
}

CLASSICAL_METHODS = list(METHOD_NAMES)[:5]

ENCODER_METHODS = list(METHOD_NAMES)[5:]

PROTOCOLS = {
    "all_text": {
        "train": ("all", "train"),
        "validation": ("all", "dev"),
        "test": ("all", "test"),
    },
    "hotels_to_medicine": {
        "train": ("hotels", "train"),
        "validation": ("hotels", "dev"),
        "test": ("medicine", "test"),
    },
}


## Pobieranie plików i sumy kontrolne

Dane pobieramy porcjami na dysk. Cały duży plik fastText nie musi
jednocześnie zajmować pamięci RAM. Suma SHA256 pozwala sprawdzić,
czy dwa uruchomienia korzystały z tych samych plików.
Raport JSON zapisujemy atomowo, aby przerwana sesja nie uszkodziła
wcześniejszych wyników.


In [ ]:
def _json_write(path, value):
    """Zapis atomowy chroni poprzedni raport, gdy sesja zostanie przerwana."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    with temporary.open("w", encoding="utf-8") as handle:
        json.dump(value, handle, ensure_ascii=False, indent=2, allow_nan=False)
    temporary.replace(path)

def _sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def _download(url, destination):
    """Pobieramy porcjami na dysk; cały duży plik nie trafia do pamięci RAM."""
    destination = Path(destination)
    if destination.is_file() and destination.stat().st_size:
        return destination
    destination.parent.mkdir(parents=True, exist_ok=True)
    partial = destination.with_suffix(destination.suffix + ".part")
    request = urllib.request.Request(url, headers={"User-Agent": "Neuronovo-PolEmo2-education/1.0"})
    print(f"Pobieranie: {destination.name}", flush=True)
    try:
        with urllib.request.urlopen(request, timeout=120) as response, partial.open("wb") as handle:
            shutil.copyfileobj(response, handle, length=1024 * 1024)
        partial.replace(destination)
    except Exception:
        partial.unlink(missing_ok=True)
        raise
    return destination


## Co porównujemy

| Doświadczenie | Zbiór uczący | Walidacja | Test |
|---|---:|---:|---:|
| `all_text`: wszystkie domeny | 6573 teksty | 823 teksty | 820 tekstów |
| `hotels_to_medicine`: hotele → medycyna | 3165 opinii o hotelach | 396 opinii o hotelach | 327 tekstów medycznych |

W drugim doświadczeniu oceniamy te same, już wybrane wagi także na
**395 hotelowych tekstach testowych** (`reference_test`). Możemy wtedy
porównać wynik dla opinii o hotelach z wynikiem dla tekstów medycznych.
Różnica wyników może zależeć również od udziałów klas i trudności tekstów.

Zachowujemy oficjalne podziały autora. Nie tworzymy nowego losowego podziału
i nie łączymy dokumentów `all_text` ze zdaniami `all_sentence`.
Każdy model dopasowujemy osobno w obu doświadczeniach. W doświadczeniu
hotelowym nie korzystamy z wag dostrojonych wcześniej w `all_text`.

Etykiety to `zero` (neutralna), `minus` (negatywna), `plus` (pozytywna)
i `amb` (mieszana/ambiwalentna). Ostatnia klasa obejmuje teksty z elementami
pozytywnymi i negatywnymi; nie muszą one występować w równych proporcjach.


In [ ]:
def load_split(domain, split, cache_dir="polemo2-cache"):
    """Czytamy format autora: tekst i etykieta w ostatnim tokenie wiersza.

    W danych występują także etykiety plus_m i minus_m. Zgodnie z loaderem
    autora sprowadzamy je do plus i minus. Nie mieszamy wariantu dokumentów
    (text) z wariantem pojedynczych zdań (sentence).
    """
    filename = f"{domain}.text.{split}.txt"
    url = f"https://huggingface.co/datasets/{DATASET_ID}/resolve/{DATASET_REVISION}/data/{filename}"
    path = _download(url, Path(cache_dir) / DATASET_REVISION / filename)
    texts, targets = [], []
    for line_number, line in enumerate(path.read_text(encoding="utf-8").splitlines(), 1):
        if not line.strip():
            continue
        parts = line.rsplit(maxsplit=1)
        if len(parts) != 2:
            raise ValueError(f"Brak etykiety: {filename}, wiersz {line_number}.")
        text, raw_label = parts
        match = re.search(r"(?:^|_)(zero|minus|plus|amb)(?:_m)?$", raw_label)
        if match is None:
            raise ValueError(f"Nieznana etykieta w {filename}, wiersz {line_number}: {raw_label!r}.")
        texts.append(text)
        targets.append(LABELS.index(match.group(1)))
    if not texts:
        raise ValueError(f"Pusty zbiór: {filename}.")
    frame = pd.DataFrame({"text": texts, "target": targets})
    metadata = {"filename": filename, "rows": len(frame), "sha256": _sha256_file(path), "source": url}
    return frame, metadata


## Miary, sprzęt i czas

Główna miara to macro-F1: średnia z F1 policzonego osobno dla każdej
klasy. Każda z czterech klas ma taką samą wagę. Dodatkowo zapisujemy
trafność (`accuracy`) i średnią czułość klas (`balanced_accuracy`).
Wyniki mieszczą się w zakresie od 0 do 1.

W raporcie zapisujemy wersje bibliotek i nazwę sprzętu. Przed pomiarem
czasu czekamy na zakończenie działań GPU, bo są wykonywane asynchronicznie.
Na GPU używamy mieszanej precyzji FP16 (ang. automatic mixed precision),
a na CPU — FP32. Nie obiecujemy konkretnego czasu działania.


In [ ]:
def _versions():
    result = {"python": platform.python_version()}
    for name in ["numpy", "pandas", "scipy", "scikit-learn", "torch", "transformers", "tokenizers", "safetensors", "huggingface-hub"]:
        try:
            result[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            result[name] = None
    return result

def _hardware():
    result = {"platform": platform.platform(), "cpu": platform.processor() or platform.machine(), "cpu_count": os.cpu_count()}
    try:
        for line in Path("/proc/cpuinfo").read_text().splitlines():
            if line.startswith("model name"):
                result["cpu"] = line.split(":", 1)[1].strip()
                break
    except OSError:
        pass
    try:
        import torch
        result["cuda_available"] = bool(torch.cuda.is_available())
        result["cuda_version"] = torch.version.cuda
        if torch.cuda.is_available():
            prop = torch.cuda.get_device_properties(0)
            result["gpu"] = prop.name
            result["gpu_memory_gib"] = round(prop.total_memory / 1024**3, 2)
    except ImportError:
        result["cuda_available"] = False
    return result

def _synchronize():
    # Operacje GPU są asynchroniczne. Bez synchronizacji czas byłby zaniżony.
    if "torch" in sys.modules:
        import torch
        if torch.cuda.is_available():
            torch.cuda.synchronize()

def _clock():
    _synchronize()
    return time.perf_counter()

def _amp_context(device):
    """Na GPU używamy mieszanego FP16; na CPU pozostajemy przy FP32."""
    import torch
    return torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == "cuda")

def _scores(y_true, y_pred):
    return {
        "macro_f1": float(f1_score(y_true, y_pred, labels=range(len(LABELS)), average="macro", zero_division=0)),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
    }


## Jak wybieramy parametry i mierzymy wynik

Zbiór **uczący** służy do wyznaczenia wag, **walidacja** do wyboru parametrów,
a **test** do końcowej oceny. Żaden klasyfikator, skaler ani słownik TF-IDF
nie jest dopasowywany do walidacji lub testu.

Główną miarą jest **macro-F1**: liczymy F1 osobno dla każdej z czterech klas,
a potem średnią. Każda klasa ma w tej średniej taką samą wagę. Zapisujemy
również odsetek poprawnych odpowiedzi (`accuracy`) i średnią czułość klas
(`balanced_accuracy`). Wszystkie wartości mieszczą się w zakresie od 0 do 1.

Dla Bayesa sprawdzamy `alpha = 0.1, 1, 10`, a dla regresji logistycznej
`C = 0.1, 1, 4`. Wygrywa najwyższe macro-F1 na walidacji; przy remisie
zostaje pierwszy parametr z tej kolejności. Najlepszą metodę także
wskazujemy według walidacji. Wynik testowy nie służy do podejmowania tych decyzji.

Plan zostaje zapisany w `protocol.json` przed oceną testową. W tym notatniku
nie zmieniaj siatki parametrów po obejrzeniu testu. Własne pomysły sprawdzaj
na treningu i walidacji; powtarzana ocena na tym samym teście przestaje
być niezależnym sprawdzianem.


In [ ]:
def _configuration():
    """Cały plan zapisujemy przed pierwszym odczytem zbioru testowego."""
    return {
        "primary_metric": "macro_f1", "seed": SEED,
        "selection": "Najwyższe macro-F1 walidacji; remis rozstrzyga kolejność siatki.",
        "tfidf": {"word_ngram_range": [1, 2], "min_df": 2, "sublinear_tf": True, "max_features": 60000},
        "char_tfidf": {"analyzer": "char_wb", "ngram_range": [3, 5], "max_features": 60000},
        "naive_bayes_alpha_grid": [0.1, 1.0, 10.0],
        "logistic_C_grid": [0.1, 1.0, 4.0],
        "logistic_max_iter": 2000, "logistic_class_weight": "balanced",
        "encoder": {"model": ENCODER_ID, "revision": ENCODER_REVISION, "max_length": 256, "batch_size": 16, "learning_rate": 2e-5, "max_epochs": 8, "patience": 2, "min_delta": 1e-4, "weight_decay": 0.01, "restore_best": True, "training_precision": "CUDA: AMP FP16 + GradScaler; CPU: FP32", "inference_precision": "CUDA: AMP FP16; CPU: FP32"},
        "fasttext": {"source": FASTTEXT_URL, "vocabulary": "Unia słów z wejściowych tekstów, bez użycia ich etykiet; wyłącznie filtr pamięci gotowego słownika", "oov": "Pominięte; brak znanych słów daje wektor zerowy"},
        "test_usage": "Etykiety testu służą wyłącznie końcowej ocenie, po wyborze parametrów danej metody. Tekst testowy może wcześniej służyć tylko do wyszukania stałego, gotowego wektora fastText; nie uczymy na nim żadnych wag ani statystyk.",
    }


## Wybór klasyfikatora na walidacji

Dla naiwnego klasyfikatora Bayesa sprawdzamy `alpha = 0.1, 1, 10`.
Dla regresji logistycznej sprawdzamy `C = 0.1, 1, 4`; mniejsze `C`
mocniej ogranicza wielkość wag. Wygrywa najwyższe macro-F1 walidacji.
Przy remisie zostaje pierwszy parametr z zapisanej kolejności.
`class_weight="balanced"` zwiększa wagę przykładów z rzadszych klas.

Każdy kandydat uczy się wyłącznie na treningu. Dla gęstych wektorów
także skaler dopasowujemy tylko do treningu. Ta funkcja nie ma dostępu
do zbioru testowego.


In [ ]:
def _linear_model(c):
    return LogisticRegression(C=c, max_iter=2000, class_weight="balanced", solver="lbfgs", random_state=SEED)

def _select_classifier(method, x_train, y_train, x_validation, y_validation, dense=False):
    """Parametr wybiera macro-F1 walidacji. Przy remisie wygrywa pierwszy.

    Każdy kandydat uczy się od początku na treningu. StandardScaler także
    dopasowujemy wyłącznie na treningu, nigdy na walidacji ani teście.
    """
    values = [0.1, 1.0, 10.0] if method == "tfidf_nb" else [0.1, 1.0, 4.0]
    best, best_score, history, fit_s, validation_s = None, -1.0, [], 0.0, 0.0
    for value in values:
        estimator = MultinomialNB(alpha=value) if method == "tfidf_nb" else _linear_model(value)
        if dense:
            estimator = make_pipeline(StandardScaler(), estimator)
        started = _clock()
        estimator.fit(x_train, y_train)
        fit_s += _clock() - started
        started = _clock()
        predicted = estimator.predict(x_validation)
        validation_s += _clock() - started
        metrics = _scores(y_validation, predicted)
        params = {"alpha" if method == "tfidf_nb" else "C": value}
        history.append({"params": params, **metrics})
        print(f"  {params}: walidacja macro-F1={metrics['macro_f1']:.4f}", flush=True)
        if metrics["macro_f1"] > best_score + 1e-12:
            best_score = metrics["macro_f1"]
            best = (estimator, predicted, metrics, params)
    return (*best, history, fit_s, validation_s)


## Zamrożony polski enkoder

Enkoder (ang. encoder) uwzględnia kontekst tokenów. Wczytujemy
`sdadas/polish-distilroberta`, wyłączamy dropout i liczenie gradientów,
a jego wagi pozostawiamy stałe. Uśredniamy reprezentacje tokenów
z pominięciem pustych miejsc wyrównujących długości partii (ang. padding).
Na tych wektorach uczymy później regresję logistyczną.

Limit wynosi 256 tokenów, nie słów. Dłuższe teksty są obcinane;
`truncation_rate` pokazuje ich udział. TF-IDF i fastText wykorzystują
całe teksty, co trzeba uwzględnić przy porównaniu.


In [ ]:
def _transformer_components(classification=False):
    import torch
    from transformers import AutoModel, AutoModelForSequenceClassification, AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained(ENCODER_ID, revision=ENCODER_REVISION)
    options = {"revision": ENCODER_REVISION, "use_safetensors": True}
    if classification:
        # Nowa czteroklasowa głowica jest losowa; wagi encodera są wstępnie nauczone.
        model = AutoModelForSequenceClassification.from_pretrained(
            ENCODER_ID, num_labels=len(LABELS), id2label=dict(enumerate(LABELS)),
            label2id={name: i for i, name in enumerate(LABELS)}, **options,
        )
    else:
        model = AutoModel.from_pretrained(ENCODER_ID, **options)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    return tokenizer, model.to(device), device

def _encoder_vectors(texts, tokenizer, model, device, batch_size=32, max_length=256):
    import torch
    model.eval()  # Wyłączamy dropout, aby te same teksty dawały te same wektory.
    vectors = []
    with torch.inference_mode():
        for offset in range(0, len(texts), batch_size):
            batch = tokenizer(texts[offset:offset + batch_size], padding=True, truncation=True, max_length=max_length, return_tensors="pt")
            batch = {key: value.to(device) for key, value in batch.items()}
            with _amp_context(device):
                hidden = model(**batch).last_hidden_state
            # Padding to puste miejsca wyrównujące długości. Nie uwzględniamy go
            # w średniej reprezentacji tokenów (ang. masked mean pooling).
            mask = batch["attention_mask"].unsqueeze(-1).to(hidden.dtype)
            pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
            # Do scikit-learn przekazujemy zwykłe wektory FP32.
            vectors.append(pooled.float().cpu().numpy())
    return np.concatenate(vectors)

def _truncation_rate(texts, tokenizer, max_length=256):
    lengths = tokenizer(texts, truncation=False, padding=False, return_length=True, verbose=False)["length"]
    return float(np.mean(np.asarray(lengths) > max_length))

def _prepare_frozen(context, row):
    """Zamroź wagi encodera i oblicz wektory treningu oraz walidacji."""
    started = _clock()
    tokenizer, encoder, device = _transformer_components()
    for parameter in encoder.parameters():
        parameter.requires_grad = False
    row['setup_s'] = _clock() - started
    started = _clock()
    x_train = _encoder_vectors(context.train_texts, tokenizer, encoder, device)
    row['feature_time'] = _clock() - started
    started = _clock()
    x_validation = _encoder_vectors(context.validation_texts, tokenizer, encoder, device)
    validation_feature_s = _clock() - started
    transform = lambda texts: _encoder_vectors(texts, tokenizer, encoder, device)
    row['truncation_rate'] = {'train': _truncation_rate(context.train_texts, tokenizer), 'validation': _truncation_rate(context.validation_texts, tokenizer)}
    row['notes'].append('Wagi encodera pozostają stałe. Uczymy regresję logistyczną na średniej reprezentacji tokenów, z pominięciem paddingu.')
    return dict(x_train=x_train, x_validation=x_validation, validation_feature_s=validation_feature_s, transform=transform, encoder=encoder, tokenizer=tokenizer, device=device)


## Przygotowanie dostrajania

Ponownie wczytujemy oryginalne wagi, dodajemy czteroklasową warstwę
wyjściową i ustawiamy ziarno losowania. Partie zawierają 16 tekstów,
tempo uczenia AdamW wynosi `2e-5`, a limit wejścia — 256 tokenów.
Ta warstwa klasyfikacyjna modelu RoBERTa różni się od regresji
logistycznej na średniej tokenów z poprzedniej metody. Porównujemy
całe metody, nie samo rozpoczęcie uczenia wag przy identycznym wyjściu.


In [ ]:
def _prepare_encoder_training(train):
    """Wczytaj pierwotne wagi, ustaw ziarno i przygotuj AdamW oraz skalowanie straty."""
    import torch
    from torch.utils.data import DataLoader
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = False
    setup_started = _clock()
    tokenizer, model, device = _transformer_components(classification=True)
    setup_s = _clock() - setup_started
    generator = torch.Generator().manual_seed(SEED)
    samples = list(zip(train.text.tolist(), train.target.tolist()))
    def collate(rows):
        texts, labels = zip(*rows)
        tokens = tokenizer(list(texts), padding=True, truncation=True, max_length=256, return_tensors='pt')
        tokens['labels'] = torch.tensor(labels, dtype=torch.long)
        return tokens
    loader = DataLoader(samples, batch_size=16, shuffle=True, generator=generator, collate_fn=collate, num_workers=0)
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-05, weight_decay=0.01)
    scaler = torch.amp.GradScaler('cuda', enabled=device.type == 'cuda')
    return (tokenizer, model, device, loader, optimizer, scaler, setup_s)


## Jedna epoka: obliczenie straty i aktualizacja wag

W tej funkcji widać cały krok uczenia: obliczenie straty, propagację
gradientów i aktualizację AdamW. Na GPU skalujemy stratę, aby małe
gradienty nie zanikały w FP16. Przed ograniczeniem normy gradientów
cofamy skalowanie (`unscale_`). Funkcja korzysta tylko z treningu.


In [ ]:
def _train_encoder_epoch(model, loader, optimizer, scaler, device):
    """Wykonaj jedną epokę uczenia; nie odczytuj walidacji ani testu."""
    import torch
    model.train()
    epoch_started = _clock()
    total_loss, total_examples = (0.0, 0)
    for batch in loader:
        batch = {key: value.to(device) for key, value in batch.items()}
        optimizer.zero_grad(set_to_none=True)
        with _amp_context(device):
            output = model(**batch)
        scaler.scale(output.loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        count = int(batch['labels'].shape[0])
        total_loss += float(output.loss.detach()) * count
        total_examples += count
    epoch_s = _clock() - epoch_started
    return (total_loss, total_examples, epoch_s)


## Walidacja, wcześniejsze zatrzymanie i najlepsze wagi

Uczymy najwyżej przez 8 epok. Po każdej mierzymy macro-F1 walidacji
i zapisujemy najlepsze wagi. Jeśli przez dwie epoki poprawa nie
przekroczy `0.0001`, kończymy uczenie (ang. early stopping,
`patience=2`). Następnie przywracamy wagi z najwyższym macro-F1
walidacji. Dopiero te wagi oceniamy na teście.

`validation_history` zachowuje przebieg uczenia, a `best_epoch`
wskazuje wybraną epokę. Wynik testowy nie wpływa na żadną z tych decyzji.


In [ ]:
def _encoder_predict(texts, tokenizer, model, device, batch_size=32):
    import torch
    model.eval()
    predictions = []
    with torch.inference_mode():
        for offset in range(0, len(texts), batch_size):
            batch = tokenizer(texts[offset:offset + batch_size], padding=True, truncation=True, max_length=256, return_tensors="pt")
            with _amp_context(device):
                logits = model(**{key: value.to(device) for key, value in batch.items()}).logits
            predictions.extend(logits.argmax(dim=-1).cpu().tolist())
    return np.asarray(predictions)

def _fit_encoder(train, validation, cache_dir):
    """Wybierz epokę po macro-F1 walidacji i przywróć jej wagi przed testem."""
    import torch
    tokenizer, model, device, loader, optimizer, scaler, setup_s = _prepare_encoder_training(train)
    checkpoint = Path(cache_dir) / 'best-encoder-state.pt'
    checkpoint.parent.mkdir(parents=True, exist_ok=True)
    best_score = patience_reference = -1.0
    best_epoch = bad_epochs = 0
    history, train_s, validation_s = ([], 0.0, 0.0)
    best_prediction, best_metrics = (None, None)
    for epoch in range(1, 9):
        total_loss, total_examples, epoch_s = _train_encoder_epoch(model, loader, optimizer, scaler, device)
        train_s += epoch_s
        started = _clock()
        predicted = _encoder_predict(validation.text.tolist(), tokenizer, model, device)
        evaluation_s = _clock() - started
        validation_s += evaluation_s
        metrics = _scores(validation.target, predicted)
        history.append({'epoch': epoch, 'loss': total_loss / total_examples, 'train_s': epoch_s, 'validation_s': evaluation_s, **metrics})
        print(f"  Epoka {epoch}/8: loss={total_loss / total_examples:.4f}, walidacja macro-F1={metrics['macro_f1']:.4f}", flush=True)
        if metrics['macro_f1'] > best_score:
            best_score, best_epoch = (metrics['macro_f1'], epoch)
            best_prediction, best_metrics = (predicted, metrics)
            torch.save({key: value.detach().cpu() for key, value in model.state_dict().items()}, checkpoint)
        if metrics['macro_f1'] > patience_reference + 0.0001:
            patience_reference, bad_epochs = (metrics['macro_f1'], 0)
        else:
            bad_epochs += 1
        if bad_epochs >= 2:
            print('  Dwie epoki bez dostatecznej poprawy: kończymy uczenie.', flush=True)
            break
    model.load_state_dict(torch.load(checkpoint, map_location='cpu', weights_only=True))
    model.to(device).eval()
    del optimizer, scaler, loader
    return (tokenizer, model, device, best_prediction, best_metrics, best_epoch, history, setup_s, train_s, validation_s)

def _run_finetuned(context, row):
    """Po przywróceniu najlepszych wag oceń test i hotelowy test odniesienia."""
    tokenizer, encoder, device, _, row['validation'], row['best_epoch'], row['validation_history'], row['setup_s'], row['train_s'], validation_s = _fit_encoder(context.train, context.validation, Path(context.cache_dir) / context.protocol)
    row['fit_s'] = row['train_s']
    row['selected_params'] = {**_configuration()['encoder'], 'best_epoch': row['best_epoch']}
    current_test = _get_test(context)
    started = _clock()
    predicted = _encoder_predict(current_test.text.tolist(), tokenizer, encoder, device)
    test_s = _clock() - started
    _evaluate_reference(context, row, lambda texts: _encoder_predict(texts, tokenizer, encoder, device))
    row['truncation_rate'] = {name: _truncation_rate(frame.text.tolist(), tokenizer) for name, frame in [('train', context.train), ('validation', context.validation), ('test', current_test)]}
    row['notes'].append('Dostrajamy wszystkie wagi. To głowica klasyfikacyjna modelu RoBERTa, a nie regresja logistyczna na średniej tokenów z poprzedniej metody.')
    del encoder, tokenizer
    return (predicted, current_test, validation_s, test_s)


## Dopasowanie klasyfikatora i końcowa predykcja

Łączymy przygotowaną reprezentację ze wspólną funkcją wyboru parametru.
Zbiór testowy trafia wyłącznie do wybranego modelu. Hotelowy test
odniesienia oceniamy tymi samymi wagami; nie dopasowujemy na nim niczego.
W każdym doświadczeniu przygotowanie reprezentacji i uczenie zaczynają
się osobno, bez przenoszenia dostrojonych wag między doświadczeniami.


In [ ]:
def _run_vector_method(context, row, method):
    """Wybierz klasyfikator po walidacji; potem oceń wybraną konfigurację."""
    if method == "fasttext_lr":
        features = _prepare_fasttext(context, row)
    elif method == "encoder_frozen_lr":
        features = _prepare_frozen(context, row)
    else:
        features = _prepare_tfidf(context, row, method)
    dense = method in {"fasttext_lr", "encoder_frozen_lr"}
    estimator, _, row["validation"], row["selected_params"], row["validation_history"], row["fit_s"], selection_predict_s = _select_classifier(
        method, features["x_train"], context.train.target,
        features["x_validation"], context.validation.target, dense=dense,
    )
    row["train_s"] = row["feature_time"] + row["fit_s"]
    validation_s = features["validation_feature_s"] + selection_predict_s
    current_test = _get_test(context)
    started = _clock()
    predicted = estimator.predict(features["transform"](current_test.text.tolist()))
    test_s = _clock() - started
    _evaluate_reference(context, row, lambda texts: estimator.predict(features["transform"](texts)))
    if method == "fasttext_lr":
        _, coverage = _average_vectors(current_test.text.tolist(), features["vectors"], features["dimension"])
        row["coverage"]["test"] = coverage
    if method == "encoder_frozen_lr":
        row["truncation_rate"]["test"] = _truncation_rate(current_test.text.tolist(), features["tokenizer"])
    return predicted, current_test, validation_s, test_s


## Zapis planu i przygotowanie doświadczenia

Tworzymy katalog wyników, zapisujemy konfigurację i wczytujemy trening
oraz walidację. Przy ponownym uruchomieniu wybranych metod można zachować
inne, wcześniej zapisane wyniki tego samego planu. Każdy wiersz zachowuje
własne informacje o sprzęcie i wersjach bibliotek.


In [ ]:
def _create_context(protocol, output_dir, methods, cache_dir):
    """Zapisz plan i wczytaj wyłącznie trening oraz walidację."""
    'Uruchom jeden protokół; zapisuj raport po zakończeniu każdej metody.\n\n    Parametry methods mogą wskazać np. CLASSICAL_METHODS lub ENCODER_METHODS.\n    Przy ponownym uruchomieniu tego samego protokołu zachowujemy wcześniejsze\n    wiersze raportu, a wskazane metody mierzymy od początku.\n    '
    if protocol not in PROTOCOLS:
        raise ValueError(f'Nieznany protokół: {protocol}. Dostępne: {list(PROTOCOLS)}')
    methods = list(METHOD_NAMES) if methods is None else list(methods)
    if not methods or len(set(methods)) != len(methods) or any((method not in METHOD_NAMES for method in methods)):
        raise ValueError('Podaj niepustą listę różnych identyfikatorów z METHOD_NAMES.')
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    specification = json.loads(json.dumps({'name': protocol, 'splits': PROTOCOLS[protocol], **_configuration()}))
    _json_write(output_dir / 'protocol.json', specification)
    data_started = _clock()
    train, train_meta = load_split(*PROTOCOLS[protocol]['train'], cache_dir)
    validation, validation_meta = load_split(*PROTOCOLS[protocol]['validation'], cache_dir)
    data_setup_s = _clock() - data_started
    for name, frame in [('train', train), ('validation', validation)]:
        if set(frame.target) != set(range(len(LABELS))):
            raise ValueError(f'Zbiór {name} nie zawiera wszystkich czterech klas.')
    report = {'status': 'running', 'run': {'date': datetime.now(timezone.utc).isoformat(), 'protocol': protocol, 'hardware': _hardware(), 'versions': _versions(), 'protocol_config': specification, 'module_sha256': ORIGINAL_MODULE_SHA256, 'data_setup_s': data_setup_s, 'data': {'dataset': DATASET_ID, 'revision': DATASET_REVISION, 'variant': 'all_text' if protocol == 'all_text' else 'hotels_text → medicine_text', 'train_rows': len(train), 'validation_rows': len(validation), 'sources': {'train': train_meta, 'validation': validation_meta}, 'labels': LABELS}, 'notes': ['Wyniki pojedynczego uruchomienia; nie są średnią z wielu losowych inicjalizacji.', 'Teksty, etykiety poszczególnych przykładów i identyfikatory wierszy nie są eksportowane.']}, 'results': []}
    existing_path = output_dir / 'results.json'
    if existing_path.exists():
        previous = json.loads(existing_path.read_text(encoding='utf-8'))
        if previous.get('run', {}).get('protocol_config') == specification and previous.get('run', {}).get('data', {}).get('revision') == DATASET_REVISION:
            report['results'] = [row for row in previous.get('results', []) if row['id'] not in methods]
    _save_report(report, output_dir)
    train_texts, validation_texts = (train.text.tolist(), validation.text.tolist())
    return SimpleNamespace(protocol=protocol, output_dir=output_dir, cache_dir=cache_dir, methods=methods, train=train, validation=validation, train_texts=train_texts, validation_texts=validation_texts, report=report, test=None, test_meta=None, reference_test=None)


## Dwa testy w doświadczeniu ze zmianą domeny

Końcowa ocena korzysta z oficjalnego testu. W doświadczeniu hotelowym
główny test obejmuje 327 tekstów medycznych, a `reference_test` —
395 tekstów o hotelach odłożonych do testu. Żaden nie służy do strojenia.
`domain_gap` oznacza wynik hotelowy minus wynik medyczny. Różnica może
zależeć także od udziałów klas i trudności tekstów.


In [ ]:
def _get_test(context):
    """Ocena odłożonych danych bez dopasowywania wag."""
    if context.test is None:
        context.test, context.test_meta = load_split(*PROTOCOLS[context.protocol]['test'], context.cache_dir)
        context.report['run']['data']['test_rows'] = len(context.test)
        context.report['run']['data']['sources']['test'] = context.test_meta
    return context.test

def _get_reference_test(context):
    """Ocena odłożonych danych bez dopasowywania wag."""
    if context.protocol != 'hotels_to_medicine':
        return None
    if context.reference_test is None:
        context.reference_test, metadata = load_split('hotels', 'test', context.cache_dir)
        context.report['run']['data']['reference_test_rows'] = len(context.reference_test)
        context.report['run']['data']['sources']['reference_test'] = metadata
    return context.reference_test

def _evaluate_reference(context, row, predict_function):
    """Ocena odłożonych danych bez dopasowywania wag."""
    reference = _get_reference_test(context)
    if reference is not None:
        started = _clock()
        predictions = predict_function(reference.text.tolist())
        row['reference_test_predict_s'] = _clock() - started
        row['reference_test'] = _scores(reference.target, predictions)


## Co znajdziesz w plikach wynikowych

Raport zapisuje wynik po każdej metodzie, także wtedy, gdy inna metoda
zakończy się błędem. Błąd ma status `error`; nie zastępujemy go liczbą.
Pełne historie walidacji, wybrane parametry i wyniki hotelowego testu
odniesienia są w JSON. CSV zawiera główne miary walidacji i testu.

`train_s` obejmuje przygotowanie reprezentacji treningowych oraz uczenie
wszystkich kandydatów sprawdzanych w siatce, a dla dostrajania — wszystkie
wykonane epoki. Nie jest to wyłącznie czas uczenia zwycięzcy.
`predict_s.validation` obejmuje ocenę kandydatów lub kolejnych epok.
`predict_s.test` mierzy pełną predykcję wybranego modelu, razem z zamianą
tekstów na wektory. Pobranie i wczytanie gotowych wag znajduje się osobno
w `setup_s`. Nie porównuj samych liczb sekund bez sprawdzenia sprzętu.

Raport zawiera wersje bibliotek, nazwę GPU/CPU, rewizje danych i modelu
oraz sumy kontrolne źródeł. Nie zawiera tekstów ani etykiet poszczególnych
przykładów. Pojedyncze uruchomienie nie pokazuje całej zmienności wyniku
wynikającej z losowej inicjalizacji i pracy GPU.


In [ ]:
def _save_report(report, output_dir):
    output_dir = Path(output_dir)
    _json_write(output_dir / "results.json", report)
    rows = []
    for row in report["results"]:
        flat = {key: row.get(key) for key in ["id", "name", "status", "train_s", "setup_s", "feature_time", "fit_s", "best_epoch"]}
        for split in ["validation", "test"]:
            for metric, value in row.get(split, {}).items():
                flat[f"{split}_{metric}"] = value
            flat[f"{split}_predict_s"] = row.get("predict_s", {}).get(split)
        flat["selected_params"] = json.dumps(row.get("selected_params", {}), ensure_ascii=False)
        rows.append(flat)
    if rows:
        pd.DataFrame(rows).to_csv(output_dir / "results.csv", index=False)

def _compact_report(report):
    """Wydruk mieści się w Colabie; pełne historie są w pliku JSON."""
    compact = copy.deepcopy(report)
    if "protocols" in compact:
        compact["protocols"] = {key: _compact_report(value) for key, value in compact["protocols"].items()}
    else:
        for row in compact["results"]:
            row.pop("validation_history", None)
    return compact

def _run_one_method(context, method):
    """Wykonaj metodę, zapisz prawdziwy wynik albo jawny błąd i zwolnij pamięć."""
    print(f'\n[{context.protocol}] {METHOD_NAMES[method]}', flush=True)
    row = {'id': method, 'name': METHOD_NAMES[method], 'status': 'running', 'setup_s': 0.0, 'feature_time': 0.0, 'fit_s': 0.0, 'truncation_rate': None, 'notes': [], 'selected_params': {}, 'validation_history': []}
    row['run_context'] = {'date': datetime.now(timezone.utc).isoformat(), 'hardware': context.report['run']['hardware'], 'versions': context.report['run']['versions'], 'module_sha256': context.report['run']['module_sha256']}
    try:
        if method == 'dummy':
            predicted, current_test, validation_s, test_s = _run_dummy(context, row)
        elif method in {'tfidf_nb', 'tfidf_lr', 'tfidf_word_char_lr', 'fasttext_lr', 'encoder_frozen_lr'}:
            predicted, current_test, validation_s, test_s = _run_vector_method(context, row, method)
        else:
            predicted, current_test, validation_s, test_s = _run_finetuned(context, row)
        row['test'] = _scores(current_test.target, predicted)
        if 'reference_test' in row:
            row['domain_gap'] = {key: row['reference_test'][key] - row['test'][key] for key in row['test']}
        row['predict_s'] = {'validation': validation_s, 'test': test_s}
        row['status'] = 'completed'
        print(f"  Wynik: val macro-F1={row['validation']['macro_f1']:.4f}; test macro-F1={row['test']['macro_f1']:.4f}", flush=True)
    except Exception as error:
        row['status'] = 'error'
        row['error'] = f'{type(error).__name__}: {error}'
        print(f"  Błąd: {row['error']}", flush=True)
    context.report['results'].append(row)
    context.report['results'].sort(key=lambda result: list(METHOD_NAMES).index(result['id']))
    _save_report(context.report, context.output_dir)
    gc.collect()
    if 'torch' in sys.modules:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


## Wykonanie planu i archiwum

Te funkcje wywołują opisane wyżej metody, łączą ich raporty i tworzą ZIP.
Najlepszą metodę wybierają według walidacji. Jeśli obliczenie zakończy się
błędem, w raporcie zostaje jawny status i komunikat. Nie wpisujemy
zastępczych ani przewidywanych wyników.


In [ ]:
def run_protocol(protocol="all_text", output_dir="polemo2-results", methods=None, cache_dir="polemo2-cache"):
    """Uruchom wybrane metody według jawnego planu jednego doświadczenia."""
    context = _create_context(protocol, output_dir, methods, cache_dir)
    for method in context.methods:
        _run_one_method(context, method)
    complete_rows = [row for row in context.report['results'] if row['status'] == 'completed']
    if complete_rows:
        best = max(complete_rows, key=lambda row: row['validation']['macro_f1'])
        context.report['run']['best_method_by_validation'] = best['id']
    context.report['status'] = 'completed' if all((row['status'] == 'completed' for row in context.report['results'])) else 'partial'
    _save_report(context.report, context.output_dir)
    return context.report

def run_all(output_dir="polemo2-results", methods=None, protocols=None, cache_dir="polemo2-cache"):
    """Uruchom oba doświadczenia i zapisz JSON, CSV oraz archiwum ZIP.

    Przykład krótkiego uruchomienia: run_all(methods=CLASSICAL_METHODS).
    Kolejne run_all(methods=ENCODER_METHODS) dopisze pozostałe metody do tych
    samych raportów, o ile konfiguracja i rewizja danych są takie same.
    """
    protocols = list(PROTOCOLS) if protocols is None else list(protocols)
    if not protocols or any(protocol not in PROTOCOLS for protocol in protocols):
        raise ValueError(f"Dostępne protokoły: {list(PROTOCOLS)}")
    output_dir = Path(output_dir)
    combined = {"status": "running", "protocols": {}}
    for protocol in protocols:
        combined["protocols"][protocol] = run_protocol(protocol, output_dir / protocol, methods=methods, cache_dir=cache_dir)
        combined["status"] = "completed" if all(report["status"] == "completed" for report in combined["protocols"].values()) else "partial"
        _json_write(output_dir / "results.json", combined)
    csv_frames = []
    for protocol in combined["protocols"]:
        csv_path = output_dir / protocol / "results.csv"
        if csv_path.exists():
            frame = pd.read_csv(csv_path)
            frame.insert(0, "protocol", protocol)
            csv_frames.append(frame)
    if csv_frames:
        pd.concat(csv_frames, ignore_index=True).to_csv(output_dir / "results.csv", index=False)
    archive = shutil.make_archive(str(output_dir.resolve()), "zip", root_dir=output_dir)
    print("\nNEURONOVO_RESULTS_JSON", flush=True)
    print(json.dumps(_compact_report(combined), ensure_ascii=False, allow_nan=False), flush=True)
    print(f"Archiwum wyników: {archive}", flush=True)
    return combined


## Uruchomienie porównania

Teraz zaczynają się obliczenia. Porównujemy dwie metody w dwóch doświadczeniach. To cztery uruchomienia metod.
Tam, gdzie przewidziano wybór parametrów, wykonujemy również kolejne dopasowania.
Postęp i wyniki walidacji pojawiają się poniżej. Zaczekaj na zakończenie
komórki. Jeśli sesja zostanie przerwana, zachowaj dostępne pliki wyników.


In [ ]:
raport = run_all(output_dir="polemo2-results", methods=ENCODER_METHODS)


## Zestawienie wyników

Walidacja i test znajdują się w osobnych kolumnach. Dla doświadczenia
hotelowego zobaczysz także wynik testu hotelowego. Najpierw sprawdź,
którą metodę wybralibyśmy według walidacji, a potem odczytaj jej wynik
testowy. Nie zakładamy z góry, która metoda okaże się najlepsza.


### Wyniki opublikowanego uruchomienia

Poniższe liczby pochodzą z zapisanego raportu. Nie są wynikami Twojej bieżącej sesji. Uruchomienie `run_all` tworzy własny raport.

**`all_text`** — data pomiaru: `2026-10-26T20:09:34.693832+00:00`; GPU: Tesla T4.

| Metoda | Macro-F1 walidacji | Macro-F1 testu | Trafność testu |
|---|---:|---:|---:|
| Najczęstsza klasa | 0.1349 | 0.1462 | 0.4134 |
| TF-IDF słów + naiwny klasyfikator Bayesa | 0.7921 | 0.8119 | 0.8244 |
| TF-IDF słów + regresja logistyczna | 0.8127 | 0.8070 | 0.8171 |
| TF-IDF słów i fragmentów słów + regresja logistyczna | 0.8276 | 0.8116 | 0.8159 |
| Średnia wektorów fastText PL + regresja logistyczna | 0.7683 | 0.7622 | 0.7646 |
| Zamrożony polski encoder + regresja logistyczna | 0.7813 | 0.7600 | 0.7610 |
| Dostrajanie polskiego encodera | 0.8431 | 0.8366 | 0.8451 |

**`hotels_to_medicine`** — data pomiaru: `2026-10-26T20:19:01.842494+00:00`; GPU: Tesla T4.

| Metoda | Macro-F1 walidacji | Macro-F1 testu | Trafność testu |
|---|---:|---:|---:|
| Najczęstsza klasa | 0.1387 | 0.1446 | 0.4067 |
| TF-IDF słów + naiwny klasyfikator Bayesa | 0.7787 | 0.4865 | 0.6239 |
| TF-IDF słów + regresja logistyczna | 0.8107 | 0.5117 | 0.6636 |
| TF-IDF słów i fragmentów słów + regresja logistyczna | 0.8270 | 0.5585 | 0.6881 |
| Średnia wektorów fastText PL + regresja logistyczna | 0.7965 | 0.5834 | 0.7064 |
| Zamrożony polski encoder + regresja logistyczna | 0.7934 | 0.6012 | 0.7064 |
| Dostrajanie polskiego encodera | 0.8128 | 0.6448 | 0.7248 |

[Pełny raport JSON z metadanymi](https://neuronovo.si/assets/projects/polemo2/results.json).


In [ ]:
import pandas as pd
from IPython.display import display

def pokaz_wyniki(raport):
    # Zestawiamy walidację i test w osobnych kolumnach.
    wiersze = []
    for protokol, czesc in raport["protocols"].items():
        for wynik in czesc["results"]:
            wiersz = {
                "doświadczenie": protokol,
                "metoda": wynik["name"],
                "status": wynik["status"],
                "F1 walidacji": wynik.get("validation", {}).get("macro_f1"),
                "F1 testu": wynik.get("test", {}).get("macro_f1"),
                "trafność testu": wynik.get("test", {}).get("accuracy"),
                "średnia czułość testu": wynik.get("test", {}).get("balanced_accuracy"),
                "uczenie [s]": wynik.get("train_s"),
            }
            if "reference_test" in wynik:
                wiersz["F1 testu hotelowego"] = wynik["reference_test"]["macro_f1"]
            if "error" in wynik:
                wiersz["błąd"] = wynik["error"]
            wiersze.append(wiersz)
    display(pd.DataFrame(wiersze).round(4))


# Własny pomiar ma pierwszeństwo. Bez niego można obejrzeć jawnie
# oznaczony raport opublikowany razem z artykułem.
if "raport" in globals():
    raport_do_podgladu = raport
    print("Wyniki obliczone w tej sesji notatnika.")
else:
    import json
    import urllib.request
    adres_wynikow = "https://neuronovo.si/assets/projects/polemo2/results.json"
    with urllib.request.urlopen(adres_wynikow) as odpowiedz:
        raport_do_podgladu = json.load(odpowiedz)
    print("Podgląd opublikowanego uruchomienia. Nie są to wyniki obliczone w tej sesji.")
pokaz_wyniki(raport_do_podgladu)


## Pobierz wyniki

Archiwum zawiera raport zbiorczy i osobne raporty doświadczeń. Pobierz
je przed odłączeniem środowiska Colab. Raporty nie zawierają tekstów
poszczególnych opinii; przykłady z ćwiczenia pozostają tylko w jego wyniku.


In [ ]:
from google.colab import files

# Archiwum zawiera raport zbiorczy i osobne raporty obu doświadczeń.
# Pobierz je przed odłączeniem maszyny Colab.
files.download("polemo2-results.zip")


## Źródła

- [PolEmo2: oficjalne repozytorium danych](https://huggingface.co/datasets/clarin-pl/polemo2-official).
- [Kocoń, Miłkowski, Zaśko-Zielińska: opis zbioru PolEmo2](https://aclanthology.org/K19-1092/).
- [fastText: wektory Wikipedii, w tym języka polskiego](https://fasttext.cc/docs/en/pretrained-vectors.html) — CC BY-SA 3.0.
- [sdadas/polish-distilroberta: karta modelu i pliki](https://huggingface.co/sdadas/polish-distilroberta) — Apache 2.0.
- [scikit-learn: TF-IDF](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html),
  [regresja logistyczna](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html),
  [macro-F1](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.f1_score.html).
- [Google Colab: zasady bezpłatnych zasobów](https://research.google.com/colaboratory/faq.html).

Kod używa dokładnie wskazanych wersji zbioru i modelu (ang. revision),
zapisanych w raporcie.
Szczegółowe warunki korzystania z danych znajdziesz w repozytorium PolEmo2.
